# Proyek Rekayasa Data Post UTS
## Implementasi Data Pipeline dan Data Warehouse


### Anggota Kelompok:

*   Adnan
*   Aston
*   Muhammad Ilkham Abdillah (24/537977/TK/59653)
*   Juan
*   Natanael Sebastian Simanjuntak (24/542676/TK/60273)
*   Rafif
*   Rakan Hendian Ramadhan (24/540158/TK/59909)
*   Rian
---
Data Source:
*   presence_kapanewon_hourly.csv
*   od_home_visits_kapanewon_monthly.csv



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive
import os

In [ ]:
!pip install duckdb pandas

import duckdb

In [ ]:
# Pastikan sudah melakukan step berikut di google drive:
# https://drive.google.com/drive/folders/1aeYsqdJvOJxurlzH-z-XCUFv0YOfhnbI?usp=sharing
# Shared With Me -> rekdat_postuts (klik kanan) -> Organize -> Add Shorcut -> Pilih My Drive -> Add
drive.mount('/content/drive')
DATA_DIR = "/content/drive/MyDrive/rekdat_postuts/data"
DB_PATH = "/content/drive/MyDrive/rekdat_postuts/db/mobility.duckdb"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# print(f"Menghubungkan ke DuckDB di: {DB_PATH}")
# con = duckdb.connect(DB_PATH)

# connect database nanti aja, transform dulu?

In [ ]:
path_presence = os.path.join(DATA_DIR, "presence_kapanewon_hourly.csv")
df_presence = pd.read_csv(path_presence)
df_presence.head()

,kab_code,kab_kota,kapanewon_code,kapanewon,date,hour,n_maids,n_pings
0,34.01,KULON PROGO,34.01.01,TEMON,2021-12-01,0,16,66
1,34.01,KULON PROGO,34.01.02,WATES,2021-12-01,0,58,101
2,34.01,KULON PROGO,34.01.03,PANJATAN,2021-12-01,0,11,32
3,34.01,KULON PROGO,34.01.04,GALUR,2021-12-01,0,9,50
4,34.01,KULON PROGO,34.01.05,LENDAH,2021-12-01,0,14,74


In [ ]:
path_od = os.path.join(DATA_DIR, "od_home_visits_kapanewon_monthly.csv")
df_od = pd.read_csv(path_od)
df_od.head()

,month,origin_kab_code,origin_kab_kota,origin_code,origin_kapanewon,dest_kab_code,dest_kab_kota,dest_code,dest_kapanewon,n_maids,reduced_quality
0,2021-12,34.01,KULON PROGO,34.01.01,TEMON,34.01,KULON PROGO,34.01.01,TEMON,569,False
1,2021-12,34.01,KULON PROGO,34.01.01,TEMON,34.01,KULON PROGO,34.01.02,WATES,85,False
2,2021-12,34.01,KULON PROGO,34.01.01,TEMON,34.01,KULON PROGO,34.01.03,PANJATAN,42,False
3,2021-12,34.01,KULON PROGO,34.01.01,TEMON,34.01,KULON PROGO,34.01.04,GALUR,16,False
4,2021-12,34.01,KULON PROGO,34.01.01,TEMON,34.01,KULON PROGO,34.01.05,LENDAH,8,False


---
## 1. Data Understanding & Profiling (Audit Data Awal)

Tahap ini bertujuan memeriksa kondisi mentah kedua dataset sebelum masuk ke tahap *cleaning*, *transform*, dan pemuatan ke Data Warehouse (`mobility.duckdb`):
1. **Struktur, Tipe Data, Missing Values, & Duplikat**
2. **Cakupan Temporal & Kelengkapan Slot (Privacy Threshold)**
3. **Konsistensi Wilayah & Relasi Kode–Nama (`kapanewon_code` vs `kapanewon`)**
4. **Distribusi Numerik & Kualitas Data (`reduced_quality`)**

### 1.1 Struktur, Tipe Data, Missing Values, & Duplikat

In [ ]:
def audit_dataframe(df, name, key_cols):
    print(f"{name}")
    print(f"Shape          : {df.shape[0]:,} baris x {df.shape[1]} kolom")
    print(f"Null values    : {int(df.isnull().sum().sum())} (per kolom: {df.isnull().sum().to_dict()})")
    print(f"Exact dupes    : {int(df.duplicated().sum())}")
    print(f"Key dupes {key_cols}: {int(df.duplicated(subset=key_cols).sum())}")
    print("Dtypes:")
    print(df.dtypes.to_string())
    print()

audit_dataframe(df_presence, "presence_kapanewon_hourly", ["kapanewon_code", "date", "hour"])
audit_dataframe(df_od, "od_home_visits_kapanewon_monthly", ["month", "origin_code", "dest_code"])

presence_kapanewon_hourly
Shape          : 230,418 baris x 8 kolom
Null values    : 0 (per kolom: {'kab_code': 0, 'kab_kota': 0, 'kapanewon_code': 0, 'kapanewon': 0, 'date': 0, 'hour': 0, 'n_maids': 0, 'n_pings': 0})
Exact dupes    : 0
Key dupes ['kapanewon_code', 'date', 'hour']: 0
Dtypes:
kab_code          float64
kab_kota           object
kapanewon_code     object
kapanewon          object
date               object
hour                int64
n_maids             int64
n_pings             int64

od_home_visits_kapanewon_monthly
Shape          : 9,894 baris x 11 kolom
Null values    : 0 (per kolom: {'month': 0, 'origin_kab_code': 0, 'origin_kab_kota': 0, 'origin_code': 0, 'origin_kapanewon': 0, 'dest_kab_code': 0, 'dest_kab_kota': 0, 'dest_code': 0, 'dest_kapanewon': 0, 'n_maids': 0, 'reduced_quality': 0})
Exact dupes    : 0
Key dupes ['month', 'origin_code', 'dest_code']: 0
Dtypes:
month                object
origin_kab_code     float64
origin_kab_kota      object
origin_code          

### 1.2 Cakupan Temporal & Kelengkapan Slot (Privacy Threshold)

In [ ]:
p_dates = pd.to_datetime(df_presence["date"])
p_month = p_dates.dt.to_period("M").astype(str)

print("Rentang tanggal presence :", p_dates.min().date(), "s.d.", p_dates.max().date(), f"({p_dates.nunique()} hari)")
print("Jam unik presence        :", sorted(df_presence["hour"].unique().tolist()))
print("Bulan unik OD            :", sorted(df_od["month"].unique().tolist()))

monthly_presence = df_presence.assign(month=p_month).groupby("month").agg(
    days=("date", "nunique"),
    kapanewon=("kapanewon_code", "nunique"),
    actual_rows=("n_maids", "count"),
    total_maids=("n_maids", "sum"),
    total_pings=("n_pings", "sum")
)
monthly_presence["expected_max_rows"] = monthly_presence["days"] * 78 * 24
monthly_presence["coverage_pct"] = (monthly_presence["actual_rows"] / monthly_presence["expected_max_rows"] * 100).round(2)
display(monthly_presence)

Rentang tanggal presence : 2021-12-01 s.d. 2022-04-30 (151 hari)
Jam unik presence        : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
Bulan unik OD            : ['2021-12', '2022-01', '2022-02', '2022-03', '2022-04']


,days,kapanewon,actual_rows,total_maids,total_pings,expected_max_rows,coverage_pct
month,,,,,,,
2021-12,31,78,54867,6445083,17977686,58032,94.55
2022-01,31,78,53833,4425220,10604631,58032,92.76
2022-02,28,78,47546,4051306,10969230,52416,90.71
2022-03,31,78,41345,972330,2781511,58032,71.25
2022-04,30,77,32827,697621,1774215,56160,58.45


### 1.3 Konsistensi Wilayah & Relasi Kode–Nama (kapanewon_code vs kapanewon)

In [ ]:
p_codes = set(df_presence["kapanewon_code"].unique())
o_codes = set(df_od["origin_code"].unique())
d_codes = set(df_od["dest_code"].unique())

print("Jumlah kapanewon_code unik (presence, origin, dest):", len(p_codes), len(o_codes), len(d_codes))
print("Semua kode wilayah identik di kedua dataset?", p_codes == o_codes == d_codes)

dim_check = pd.concat([
    df_presence[["kab_code", "kab_kota", "kapanewon_code", "kapanewon"]].drop_duplicates(),
    df_od[["origin_kab_code", "origin_kab_kota", "origin_code", "origin_kapanewon"]].rename(
        columns={"origin_kab_code": "kab_code", "origin_kab_kota": "kab_kota", "origin_code": "kapanewon_code", "origin_kapanewon": "kapanewon"}
    ).drop_duplicates()
]).drop_duplicates()

dup_names = dim_check[dim_check.duplicated(subset=["kapanewon"], keep=False)].sort_values("kapanewon")
print(f"Total pasangan (kapanewon_code, kapanewon) unik: {len(dim_check)}")
print("Nama kapanewon yang memiliki lebih dari 1 kode wilayah:")
display(dup_names)

Jumlah kapanewon_code unik (presence, origin, dest): 78 78 78
Semua kode wilayah identik di kedua dataset? True
Total pasangan (kapanewon_code, kapanewon) unik: 78
Nama kapanewon yang memiliki lebih dari 1 kode wilayah:


,kab_code,kab_kota,kapanewon_code,kapanewon
19,34.02,BANTUL,34.02.09,JETIS
54,34.71,YOGYAKARTA,34.71.02,JETIS


### 1.4 Distribusi Numerik, Kualitas Data (reduced_quality), & Ringkasan Awal

In [ ]:
print("Statistik Numerik Presence")
p_stats = df_presence[["n_maids", "n_pings"]].assign(pings_per_maid=df_presence["n_pings"] / df_presence["n_maids"]).describe().round(2)
display(p_stats)

print("Statistik Numerik OD Home Visits")
display(df_od[["n_maids"]].describe().round(2))

print("Sebaran reduced_quality per Bulan")
display(pd.crosstab(df_od["month"], df_od["reduced_quality"], margins=True))

print("Perbandingan Intra-Kapanewon vs Inter-Kapanewon (OD)")
od_flow = df_od.assign(flow_type=np.where(df_od["origin_code"] == df_od["dest_code"], "Intra-Kapanewon", "Inter-Kapanewon"))
display(od_flow.groupby("flow_type").agg(
    rows=("n_maids", "count"),
    total_maids=("n_maids", "sum"),
    mean_maids=("n_maids", "mean"),
    median_maids=("n_maids", "median")
).round(2))

Statistik Numerik Presence


,n_maids,n_pings,pings_per_maid
count,230418.00,230418.00,230418.00
mean,72.01,191.42,2.61
std,124.18,369.30,1.15
min,5.00,5.00,1.00
25%,12.00,28.00,1.88
50%,29.00,72.00,2.39
75%,77.00,201.00,3.05
max,2506.00,10210.00,39.38


Statistik Numerik OD Home Visits


,n_maids
count,9894.00
mean,64.93
std,278.16
min,5.00
25%,8.00
50%,15.00
75%,38.00
max,7961.00


Sebaran reduced_quality per Bulan


reduced_quality,False,True,All
month,,,
2021-12,2595,0,2595
2022-01,2624,0,2624
2022-02,2653,0,2653
2022-03,0,1202,1202
2022-04,0,820,820
All,7872,2022,9894


Perbandingan Intra-Kapanewon vs Inter-Kapanewon (OD)


,rows,total_maids,mean_maids,median_maids
flow_type,,,,
Inter-Kapanewon,9504,385908,40.60,14.0
Intra-Kapanewon,390,256498,657.69,208.5


---
## 2. Data Cleaning & Transformation

Berdasarkan hasil audit di Section 1, data tidak memiliki *missing value* maupun duplikat, sehingga fokus transformasi adalah:
1. **Standarisasi & Feature Engineering `presence`** (parsing tanggal, ekstraksi fitur waktu & `time_slot`, serta metrik `pings_per_maid`).
2. **Standarisasi & Klasifikasi Aliran `od_home_visits`** (klasifikasi `flow_scope`: Intra-Kapanewon, Antar-Kapanewon dalam Kab/Kota, dan Lintas Kab/Kota, serta flag kualitas).

### 2.1 Transformasi `df_presence` -> `df_presence_clean`

In [ ]:
df_presence_clean = df_presence.copy()

# Standarisasi kode & nama wilayah
for col in ["kab_code", "kapanewon_code"]:
    df_presence_clean[col] = df_presence_clean[col].astype(str).str.strip()
for col in ["kab_kota", "kapanewon"]:
    df_presence_clean[col] = df_presence_clean[col].astype(str).str.strip().str.upper()

# Parsing tanggal & ekstraksi fitur waktu
df_presence_clean["date"] = pd.to_datetime(df_presence_clean["date"])
df_presence_clean["month"] = df_presence_clean["date"].dt.to_period("M").astype(str)
df_presence_clean["day_of_week"] = df_presence_clean["date"].dt.dayofweek  # 0=Senin, 6=Minggu
df_presence_clean["day_name"] = df_presence_clean["date"].dt.day_name()
df_presence_clean["is_weekend"] = df_presence_clean["day_of_week"] >= 5

# Pengelompokan jam aktivitas (time_slot)
df_presence_clean["time_slot"] = pd.cut(
    df_presence_clean["hour"],
    bins=[-1, 5, 9, 14, 18, 23],
    labels=["00-05 Dini Hari", "06-09 Pagi", "10-14 Siang", "15-18 Sore", "19-23 Malam"]
).astype(str)

# Metrik turunan: rata-rata ping per perangkat
df_presence_clean["pings_per_maid"] = (df_presence_clean["n_pings"] / df_presence_clean["n_maids"]).round(4)

print("Shape df_presence_clean:", df_presence_clean.shape)
display(df_presence_clean.head())

Shape df_presence_clean: (230418, 14)


,kab_code,kab_kota,kapanewon_code,kapanewon,date,hour,n_maids,n_pings,month,day_of_week,day_name,is_weekend,time_slot,pings_per_maid
0,34.01,KULON PROGO,34.01.01,TEMON,2021-12-01,0,16,66,2021-12,2,Wednesday,False,00-05 Dini Hari,4.1250
1,34.01,KULON PROGO,34.01.02,WATES,2021-12-01,0,58,101,2021-12,2,Wednesday,False,00-05 Dini Hari,1.7414
2,34.01,KULON PROGO,34.01.03,PANJATAN,2021-12-01,0,11,32,2021-12,2,Wednesday,False,00-05 Dini Hari,2.9091
3,34.01,KULON PROGO,34.01.04,GALUR,2021-12-01,0,9,50,2021-12,2,Wednesday,False,00-05 Dini Hari,5.5556
4,34.01,KULON PROGO,34.01.05,LENDAH,2021-12-01,0,14,74,2021-12,2,Wednesday,False,00-05 Dini Hari,5.2857


### 2.2 Transformasi `df_od` -> `df_od_clean`

In [ ]:
df_od_clean = df_od.copy()

# Standarisasi kode & nama wilayah
code_cols = ["origin_kab_code", "origin_code", "dest_kab_code", "dest_code"]
name_cols = ["origin_kab_kota", "origin_kapanewon", "dest_kab_kota", "dest_kapanewon"]
for col in code_cols:
    df_od_clean[col] = df_od_clean[col].astype(str).str.strip()
for col in name_cols:
    df_od_clean[col] = df_od_clean[col].astype(str).str.strip().str.upper()

df_od_clean["month"] = df_od_clean["month"].astype(str).str.strip()
df_od_clean["reduced_quality"] = df_od_clean["reduced_quality"].astype(bool)
df_od_clean["is_full_quality"] = ~df_od_clean["reduced_quality"]

# Klasifikasi cakupan pergerakan (flow_scope)
df_od_clean["flow_scope"] = np.select(
    [
        df_od_clean["origin_code"] == df_od_clean["dest_code"],
        df_od_clean["origin_kab_code"] == df_od_clean["dest_kab_code"],
    ],
    [
        "Intra-Kapanewon",
        "Antar-Kapanewon (Dalam Kab/Kota)",
    ],
    default="Lintas Kab/Kota",
)

print("Shape df_od_clean:", df_od_clean.shape)
print("\nDistribusi flow_scope:")
display(df_od_clean.groupby("flow_scope").agg(rows=("n_maids", "count"), total_maids=("n_maids", "sum")))
display(df_od_clean.head())

Shape df_od_clean: (9894, 13)

Distribusi flow_scope:


,rows,total_maids
flow_scope,,
Antar-Kapanewon (Dalam Kab/Kota),3228,188472
Intra-Kapanewon,390,256498
Lintas Kab/Kota,6276,197436


,month,origin_kab_code,origin_kab_kota,origin_code,origin_kapanewon,dest_kab_code,dest_kab_kota,dest_code,dest_kapanewon,n_maids,reduced_quality,is_full_quality,flow_scope
0,2021-12,34.01,KULON PROGO,34.01.01,TEMON,34.01,KULON PROGO,34.01.01,TEMON,569,False,True,Intra-Kapanewon
1,2021-12,34.01,KULON PROGO,34.01.01,TEMON,34.01,KULON PROGO,34.01.02,WATES,85,False,True,Antar-Kapanewon (Dalam Kab/Kota)
2,2021-12,34.01,KULON PROGO,34.01.01,TEMON,34.01,KULON PROGO,34.01.03,PANJATAN,42,False,True,Antar-Kapanewon (Dalam Kab/Kota)
3,2021-12,34.01,KULON PROGO,34.01.01,TEMON,34.01,KULON PROGO,34.01.04,GALUR,16,False,True,Antar-Kapanewon (Dalam Kab/Kota)
4,2021-12,34.01,KULON PROGO,34.01.01,TEMON,34.01,KULON PROGO,34.01.05,LENDAH,8,False,True,Antar-Kapanewon (Dalam Kab/Kota)
